# FrozenLake solved with Policy Iteration using Linear Algebra

This notebook shows how to solve a small MDP (FrozenLake) **exactly** using **policy iteration**,
where the policy evaluation step is done via a **single linear system solve**:

$$
(I - \gamma P^\pi) V^\pi = r^\pi
$$

No LP solvers, no iterative Bellman backups — just linear algebra + argmax.


In [ ]:
!pip -q install gymnasium numpy

In [ ]:
import numpy as np
import gymnasium as gym

env = gym.make("FrozenLake-v1", map_name="4x4", is_slippery=False)
P = env.unwrapped.P
nS, nA = env.observation_space.n, env.action_space.n
gamma = 0.99

nS, nA

## Policy evaluation as a linear system

For a fixed policy $\pi$:

$$
V^\pi = r^\pi + \gamma P^\pi V^\pi
\quad\Rightarrow\quad
(I-\gamma P^\pi)V^\pi = r^\pi
$$


In [ ]:
def build_Ppi_rpi(P, pi):
    Ppi = np.zeros((nS, nS))
    rpi = np.zeros(nS)
    for s in range(nS):
        a = int(pi[s])
        for (p, sp, r, terminated) in P[s][a]:
            rpi[s] += p * r
            if not terminated:
                Ppi[s, sp] += p
    return Ppi, rpi

def eval_policy_linear(P, pi, gamma):
    Ppi, rpi = build_Ppi_rpi(P, pi)
    A = np.eye(nS) - gamma * Ppi
    V = np.linalg.solve(A, rpi)
    return V

## Compute Q from V

$$
Q(s,a)=r(s,a)+\gamma \sum_{s'} P(s'|s,a)V(s')
$$


In [ ]:
def Q_from_V(P, V, gamma):
    Q = np.zeros((nS, nA))
    for s in range(nS):
        for a in range(nA):
            q = 0.0
            for (p, sp, r, terminated) in P[s][a]:
                q += p * (r + (0 if terminated else gamma * V[sp]))
            Q[s, a] = q
    return Q

## Policy iteration loop

In [ ]:
def policy_iteration(P, gamma=0.99, max_iter=100):
    pi = np.zeros(nS, dtype=int)
    for it in range(max_iter):
        V = eval_policy_linear(P, pi, gamma)
        Q = Q_from_V(P, V, gamma)
        pi_new = Q.argmax(axis=1)
        if np.array_equal(pi_new, pi):
            return pi, V, Q, it + 1
        pi = pi_new
    return pi, V, Q, max_iter

pi_star, V_star, Q_star, iters = policy_iteration(P, gamma)
iters, pi_star

## Visualisation

In [ ]:
def render_grid(arr, title=""):
    side = int(np.sqrt(arr.size))
    print(title)
    for i in range(side):
        row = arr[i*side:(i+1)*side]
        if arr.dtype.kind in {'U','O'}:
            print(" ".join(f"{x:^5}" for x in row))
        else:
            print(" ".join(f"{x:6.3f}" for x in row))
    print()

render_grid(V_star, "V*")

symbols = {0:'←', 1:'↓', 2:'→', 3:'↑'}
pi_symbols = np.array([symbols[a] for a in pi_star], dtype=object)
render_grid(pi_symbols, "π*")

## Rollout check

In [ ]:
def run_episode(env, pi, seed=None, max_steps=200):
    s, _ = env.reset(seed=seed)
    total = 0.0
    for _ in range(max_steps):
        a = int(pi[s])
        s, r, terminated, truncated, _ = env.step(a)
        total += r
        if terminated or truncated:
            break
    return total

returns = [run_episode(env, pi_star, seed=i) for i in range(200)]
print("Average return:", np.mean(returns))
print("Success rate:", np.mean(np.array(returns) > 0))